# 🔬 Transformer Internals: Token Journey Through an Encoder-Decoder Model

**What this notebook does:**
- Downloads **T5-small** (a real encoder-decoder transformer with 60M learned parameters)
- **Extracts the actual learned weights** (Q, K, V, FFN, embeddings, LM head)
- Traces a single input through **every operation**, showing tensor shapes and values at each step
- Covers: **Tokenization → Embedding → Encoder (Self-Attention + FFN) → Decoder (Masked Self-Attn + Cross-Attn + FFN) → LM Head → Predicted Tokens**

> Run each cell top-to-bottom. Every section is self-contained and prints what the tensors look like.


---
## 1. Setup & Load Model

We use **T5-small** because:
- It's a clean encoder-decoder transformer (the architecture the original "Attention Is All You Need" paper describes)
- Small enough to inspect (60M params) but real enough to produce meaningful outputs
- Widely used for translation, summarization, Q&A


In [ ]:
import torch
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration, T5Tokenizer
import warnings
warnings.filterwarnings("ignore")

# Load model and tokenizer
model_name = "t5-small"
print(f"Loading {model_name}...")
tokenizer = T5Tokenizer.from_pretrained(model_name, legacy=True)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.eval()  # inference mode (no dropout)

print(f"✓ Model loaded: {model_name}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")


---
## 2. Model Architecture Overview

Let's look at the exact structure — every weight matrix, every layer.


In [ ]:
config = model.config
print("=" * 70)
print("T5-SMALL CONFIGURATION")
print("=" * 70)
print(f"""
  d_model (hidden size):    {config.d_model}
  d_ff (feedforward dim):   {config.d_ff}
  num_heads:                {config.num_heads}
  d_kv (dim per head):      {config.d_kv}
  num_encoder_layers:       {config.num_layers}
  num_decoder_layers:       {config.num_decoder_layers}
  vocab_size:               {config.vocab_size}
""")

print("=" * 70)
print("PARAMETER BREAKDOWN")
print("=" * 70)

# Count params by component
embed_params = model.shared.weight.numel()
enc_params = sum(p.numel() for n, p in model.encoder.named_parameters()
                 if "embed_tokens" not in n)
dec_params = sum(p.numel() for n, p in model.decoder.named_parameters()
                 if "embed_tokens" not in n)
lm_params = model.lm_head.weight.numel()

print(f"  Shared Embedding:    {embed_params:>12,}  ({config.vocab_size} × {config.d_model})")
print(f"  Encoder:             {enc_params:>12,}  (6 layers × self-attn + FFN)")
print(f"  Decoder:             {dec_params:>12,}  (6 layers × self-attn + cross-attn + FFN)")
print(f"  LM Head:             {lm_params:>12,}  ({config.d_model} × {config.vocab_size})")
print(f"  {'─' * 45}")
total = embed_params + enc_params + dec_params + lm_params
print(f"  Total:               {total:>12,}")
print(f"\n  Note: T5 ties embedding and LM head weights (same matrix used for both!)")


In [ ]:
# Show the actual weight matrices in encoder layer 0
print("=" * 70)
print("ENCODER LAYER 0 — WEIGHT SHAPES")
print("=" * 70)

enc_block = model.encoder.block[0]
for name, param in enc_block.named_parameters():
    print(f"  {name:<55} {str(list(param.shape)):>20}")

print(f"\n{'=' * 70}")
print("DECODER LAYER 0 — WEIGHT SHAPES")
print("=" * 70)

dec_block = model.decoder.block[0]
for name, param in dec_block.named_parameters():
    print(f"  {name:<55} {str(list(param.shape)):>20}")


---
## 3. Tokenization: Text → Token IDs

The first step in any transformer: convert text to integer token IDs using a learned vocabulary (SentencePiece for T5).


In [ ]:
# Our example: English to German translation
input_text = "translate English to German: The house is wonderful."
target_text = "Das Haus ist wunderbar."

print("=" * 70)
print("TOKENIZATION")
print("=" * 70)

# Tokenize input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
print(f"\nInput text:    \"{input_text}\"")
print(f"Token IDs:     {input_ids[0].tolist()}")
print(f"Tokens:        {tokenizer.convert_ids_to_tokens(input_ids[0])}")
print(f"Shape:         {input_ids.shape}  (batch=1, seq_len={input_ids.shape[1]})")

# Tokenize target (for decoder input during teaching)
target_ids = tokenizer(target_text, return_tensors="pt").input_ids
print(f"\nTarget text:   \"{target_text}\"")
print(f"Token IDs:     {target_ids[0].tolist()}")
print(f"Tokens:        {tokenizer.convert_ids_to_tokens(target_ids[0])}")
print(f"Shape:         {target_ids.shape}")

# Show the vocab
print(f"\nVocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"Special tokens:  PAD={tokenizer.pad_token_id}, EOS={tokenizer.eos_token_id}")


---
## 4. Token Embedding: IDs → Vectors

Each token ID is looked up in a learned embedding table. This converts discrete IDs into continuous vectors that the model can process.

```
Token ID 37 → look up row 37 of embedding matrix → 512-dim vector
```

T5 does NOT use positional embeddings (sin/cos or learned). Instead, it uses **relative position bias** inside the attention mechanism.


In [ ]:
# Get the shared embedding matrix
embedding_matrix = model.shared.weight  # (vocab_size, d_model) = (32128, 512)
print("=" * 70)
print("EMBEDDING LAYER")
print("=" * 70)
print(f"\nEmbedding matrix shape: {list(embedding_matrix.shape)}")
print(f"  = {embedding_matrix.shape[0]:,} tokens × {embedding_matrix.shape[1]} dimensions")

# Look up embeddings for our input
with torch.no_grad():
    input_embeddings = model.shared(input_ids)  # (1, seq_len, 512)

print(f"\nInput IDs shape:        {input_ids.shape}")
print(f"After embedding shape:  {input_embeddings.shape}")
print(f"  = (batch={input_embeddings.shape[0]}, seq_len={input_embeddings.shape[1]}, d_model={input_embeddings.shape[2]})")

# Show the actual embedding values for the first token
print(f"\n{'─' * 70}")
print(f"Embedding vector for token \"{tokenizer.decode(input_ids[0, 0])}\" (ID={input_ids[0, 0].item()}):")
print(f"  First 20 values: {input_embeddings[0, 0, :20].tolist()}")
print(f"  Mean: {input_embeddings[0, 0].mean().item():.6f}")
print(f"  Std:  {input_embeddings[0, 0].std().item():.6f}")
print(f"  Norm: {input_embeddings[0, 0].norm().item():.4f}")


---
## 5. Encoder Self-Attention (Layer 0) — Step by Step

This is the core operation. We'll manually extract Q, K, V weight matrices from the model and compute attention by hand.

**T5 uses pre-layer-norm**: the input is normalized BEFORE attention (not after).

```
Input x
  │
  ├── LayerNorm(x) ──→ Q = norm_x @ W_Q   ──→ reshape into 8 heads
  │                ──→ K = norm_x @ W_K   ──→ reshape into 8 heads
  │                ──→ V = norm_x @ W_V   ──→ reshape into 8 heads
  │                         │
  │                    Attention(Q, K, V) + position bias
  │                         │
  │                    Concat heads → W_O projection
  │                         │
  └── + (residual) ─────────┘
  │
  Output
```


In [ ]:
print("=" * 70)
print("ENCODER LAYER 0 — SELF-ATTENTION (MANUAL COMPUTATION)")
print("=" * 70)

enc_layer0 = model.encoder.block[0]

# ─── Step 1: Layer Norm ───
layer_norm = enc_layer0.layer[0].layer_norm
normed = layer_norm(input_embeddings)

print(f"\n─── Step 1: RMS Layer Norm ───")
print(f"  Input:      {list(input_embeddings.shape)} → mean={input_embeddings[0, 0].mean():.4f}, std={input_embeddings[0, 0].std():.4f}")
print(f"  After norm: {list(normed.shape)} → mean={normed[0, 0].mean():.4f}, std={normed[0, 0].std():.4f}")
print(f"  (Normalization stabilizes the values before attention)")


In [ ]:
# ─── Step 2: Extract Q, K, V weight matrices ───
print("─── Step 2: Q, K, V Projection ───")

attn = enc_layer0.layer[0].SelfAttention

W_Q = attn.q.weight  # (512, 512)
W_K = attn.k.weight
W_V = attn.v.weight
W_O = attn.o.weight

print(f"  W_Q shape: {list(W_Q.shape)}  (d_model → num_heads × d_kv = 8 × 64 = 512)")
print(f"  W_K shape: {list(W_K.shape)}")
print(f"  W_V shape: {list(W_V.shape)}")
print(f"  W_O shape: {list(W_O.shape)}  (num_heads × d_kv → d_model)")

# Compute Q, K, V
# Note: T5 Linear layers have no bias
with torch.no_grad():
    Q = normed @ W_Q.T  # (1, seq_len, 512)
    K = normed @ W_K.T
    V = normed @ W_V.T

print(f"\n  Q = LayerNorm(x) @ W_Q^T → shape: {list(Q.shape)}")
print(f"  K = LayerNorm(x) @ W_K^T → shape: {list(K.shape)}")
print(f"  V = LayerNorm(x) @ W_V^T → shape: {list(V.shape)}")


In [ ]:
# ─── Step 3: Reshape into multiple heads ───
print("─── Step 3: Reshape into 8 Heads ───")

batch_size = 1
seq_len = input_ids.shape[1]
num_heads = 8
d_kv = 64  # 512 / 8

Q_heads = Q.view(batch_size, seq_len, num_heads, d_kv).transpose(1, 2)
K_heads = K.view(batch_size, seq_len, num_heads, d_kv).transpose(1, 2)
V_heads = V.view(batch_size, seq_len, num_heads, d_kv).transpose(1, 2)

print(f"  Q: {list(Q.shape)} → {list(Q_heads.shape)}  (batch, heads, seq, d_kv)")
print(f"  K: {list(K.shape)} → {list(K_heads.shape)}")
print(f"  V: {list(V.shape)} → {list(V_heads.shape)}")
print(f"\n  Each of the 8 heads now operates on 64-dim subspace independently")


In [ ]:
# ─── Step 4: Compute attention scores ───
print("─── Step 4: Attention Scores (Q · K^T) ───")

import math

scores = torch.matmul(Q_heads, K_heads.transpose(-2, -1))
print(f"  Q @ K^T shape: {list(scores.shape)}  (batch, heads, seq_q, seq_k)")
print(f"  This is the {seq_len}×{seq_len} attention matrix for each of {num_heads} heads")

# T5 does NOT scale by sqrt(d_k)! It uses unscaled attention.
# (The scaling is absorbed into the initialization of the weights)
print(f"\n  ⚠  T5 does NOT divide by √d_k!")
print(f"     Unlike standard attention, T5 absorbs scaling into weight initialization.")

# T5 also adds relative position bias (from the first layer)
rel_bias_embed = enc_layer0.layer[0].SelfAttention.relative_attention_bias
print(f"\n  Relative position bias embedding: {list(rel_bias_embed.weight.shape)}")
print(f"    = 32 distance buckets × {num_heads} heads")
print(f"    (Encodes 'how far apart are these tokens' without absolute positions)")

# Show raw score statistics
print(f"\n  Raw attention scores (head 0, before softmax):")
print(f"    min={scores[0, 0].min():.4f}, max={scores[0, 0].max():.4f}, mean={scores[0, 0].mean():.4f}")


In [ ]:
# ─── Step 5: Softmax → attention weights ───
print("─── Step 5: Softmax → Attention Weights ───")

weights = F.softmax(scores, dim=-1)

print(f"  Attention weights shape: {list(weights.shape)}")
print(f"  Each row sums to 1: {weights[0, 0, 0].sum().item():.6f}")

# Show the attention pattern for head 0
print(f"\n  Attention weights for HEAD 0:")
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
# Print header
header = "  " + " ".join(f"{t[:6]:>7}" for t in tokens)
print(header)
for i, row in enumerate(weights[0, 0].detach().numpy()):
    row_str = f"  {tokens[i][:6]:>6} " + " ".join(f"{v:>7.3f}" for v in row)
    print(row_str)

print(f"\n  → Each row shows HOW MUCH each token attends to every other token")
print(f"    (These are the LEARNED patterns from pre-training on ~750GB of text)")


In [ ]:
# ─── Step 6: Weighted sum of Values ───
print("─── Step 6: Weighted Sum of Values ───")

attn_output = torch.matmul(weights, V_heads)
print(f"  weights @ V shape: {list(attn_output.shape)}  (batch, heads, seq, d_kv)")

# ─── Step 7: Concatenate heads + output projection ───
print(f"\n─── Step 7: Concatenate Heads + Output Projection ───")

# (batch, heads, seq, d_kv) → (batch, seq, heads, d_kv) → (batch, seq, d_model)
concat = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, num_heads * d_kv)
print(f"  After concat: {list(concat.shape)}")

projected = concat @ W_O.T
print(f"  After W_O:    {list(projected.shape)}")

# ─── Step 8: Residual connection ───
print(f"\n─── Step 8: Residual Connection ───")
after_attn = input_embeddings + projected
print(f"  Output = input + attention_output")
print(f"  Shape: {list(after_attn.shape)}")
print(f"  This residual connection is CRITICAL — it lets gradients flow directly")
print(f"  through the network during training (skip connection).")


---
## 6. Encoder Feed-Forward Network (Layer 0)

After self-attention, each position is processed independently through a 2-layer FFN:

```
LayerNorm(x) → Linear(512 → 2048) → ReLU → Linear(2048 → 512) → + residual
```

The FFN processes each token INDEPENDENTLY (no interaction between positions). Attention handles inter-token relationships; FFN handles per-token transformations.


In [ ]:
print("=" * 70)
print("ENCODER LAYER 0 — FEED-FORWARD NETWORK")
print("=" * 70)

ffn = enc_layer0.layer[1]
ffn_norm = ffn.layer_norm
W_up = ffn.DenseReluDense.wi.weight    # (2048, 512) — expand
W_down = ffn.DenseReluDense.wo.weight  # (512, 2048) — compress back

print(f"\n  W_up shape:   {list(W_up.shape)}   (512 → 2048: expand to 4× wider)")
print(f"  W_down shape: {list(W_down.shape)}   (2048 → 512: compress back)")

with torch.no_grad():
    # Layer norm
    ffn_normed = ffn_norm(after_attn)
    print(f"\n  After LayerNorm: {list(ffn_normed.shape)}")

    # Up-project + ReLU
    hidden = F.relu(ffn_normed @ W_up.T)
    print(f"  After W_up + ReLU: {list(hidden.shape)}  ← 4× wider!")

    # Count how many neurons are active (non-zero after ReLU)
    active = (hidden > 0).float().mean().item()
    print(f"    ReLU sparsity: {active:.1%} neurons active (rest are zeroed out)")

    # Down-project
    ffn_output = hidden @ W_down.T
    print(f"  After W_down:    {list(ffn_output.shape)}  ← back to d_model")

    # Residual
    encoder_layer0_output = after_attn + ffn_output
    print(f"\n  Final output (with residual): {list(encoder_layer0_output.shape)}")
    print(f"  Values: mean={encoder_layer0_output[0].mean():.4f}, std={encoder_layer0_output[0].std():.4f}")


---
## 7. Full Encoder: All 6 Layers

Now let's run through ALL encoder layers and watch how the tensor evolves. We use the model's actual forward pass here.


In [ ]:
print("=" * 70)
print("FULL ENCODER — TENSOR EVOLUTION ACROSS ALL 6 LAYERS")
print("=" * 70)

with torch.no_grad():
    # Run the encoder properly
    encoder_outputs = model.encoder(
        input_ids=input_ids,
        output_hidden_states=True,  # get intermediate states
        output_attentions=True,     # get attention weights
    )

print(f"\nEncoder received {input_ids.shape[1]} tokens, output {encoder_outputs.last_hidden_state.shape[1]} vectors")

# Show how the representation evolves through layers
print(f"\n{'Layer':<10} {'Shape':<25} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("─" * 75)

for i, hidden in enumerate(encoder_outputs.hidden_states):
    h = hidden[0]  # batch=0
    label = "Embed" if i == 0 else f"Layer {i}"
    print(f"  {label:<8} {str(list(hidden.shape)):<25} {h.mean():>10.4f} {h.std():>10.4f} {h.min():>10.4f} {h.max():>10.4f}")

print(f"\n  The representation gets progressively more refined through each layer.")
print(f"  By layer 6, the embeddings encode rich contextual meaning for each token.")


In [ ]:
# Show how attention patterns differ across layers
print("=" * 70)
print("ATTENTION PATTERNS ACROSS ENCODER LAYERS")
print("=" * 70)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

for layer_idx in [0, 2, 5]:  # First, middle, last
    attn = encoder_outputs.attentions[layer_idx][0]  # (heads, seq, seq)
    print(f"\nLayer {layer_idx} — Where does each token focus? (argmax per head)")
    print(f"  {'Token':<15}", end="")
    for h in range(min(4, attn.shape[0])):
        print(f"  Head {h:<3}", end="")
    print()
    
    for t_idx, tok in enumerate(tokens):
        print(f"  {tok:<15}", end="")
        for h in range(min(4, attn.shape[0])):
            max_idx = attn[h, t_idx].argmax().item()
            print(f"  → {tokens[max_idx][:5]:<5}", end="")
        print()


---
## 8. Decoder: Preparing Inputs

The decoder works **autoregressively**: it generates one token at a time, feeding each predicted token back as input.

For teacher forcing (training), we provide the target tokens shifted right:
```
Target:       "Das Haus ist wunderbar."
Decoder in:   <pad>  Das  Haus  ist  wunderbar  .
Decoder out:   Das   Haus ist   wunderbar .      </s>
```


In [ ]:
print("=" * 70)
print("DECODER INPUT PREPARATION")
print("=" * 70)

# Prepare decoder input (shift target right, prepend pad token)
decoder_input_ids = model._shift_right(target_ids)

print(f"\nTarget tokens:        {tokenizer.convert_ids_to_tokens(target_ids[0])}")
print(f"Target IDs:           {target_ids[0].tolist()}")
print(f"\nDecoder input tokens: {tokenizer.convert_ids_to_tokens(decoder_input_ids[0])}")
print(f"Decoder input IDs:    {decoder_input_ids[0].tolist()}")
print(f"\n  → The decoder input is the target shifted right by 1 position")
print(f"    so the model learns to predict the NEXT token at each position.")

# Embed decoder inputs
with torch.no_grad():
    decoder_embeddings = model.decoder.embed_tokens(decoder_input_ids)

print(f"\nDecoder embedding shape: {list(decoder_embeddings.shape)}")


---
## 9. Decoder Masked Self-Attention (Layer 0)

The decoder uses **causal masking**: each position can only attend to earlier positions (can't look at future tokens it hasn't generated yet).

```
            Position 0  1  2  3  4
Position 0:    ✓       ✗  ✗  ✗  ✗
Position 1:    ✓       ✓  ✗  ✗  ✗
Position 2:    ✓       ✓  ✓  ✗  ✗
Position 3:    ✓       ✓  ✓  ✓  ✗
Position 4:    ✓       ✓  ✓  ✓  ✓
```


In [ ]:
print("=" * 70)
print("DECODER LAYER 0 — MASKED SELF-ATTENTION")
print("=" * 70)

dec_layer0 = model.decoder.block[0]
dec_attn = dec_layer0.layer[0].SelfAttention

# Extract decoder Q, K, V weights
W_Q_dec = dec_attn.q.weight
W_K_dec = dec_attn.k.weight
W_V_dec = dec_attn.v.weight

print(f"\n  Decoder Self-Attention weights:")
print(f"    W_Q: {list(W_Q_dec.shape)}")
print(f"    W_K: {list(W_K_dec.shape)}")
print(f"    W_V: {list(W_V_dec.shape)}")

# Compute Q, K, V for decoder
with torch.no_grad():
    dec_normed = dec_layer0.layer[0].layer_norm(decoder_embeddings)
    Q_dec = dec_normed @ W_Q_dec.T
    K_dec = dec_normed @ W_K_dec.T
    V_dec = dec_normed @ W_V_dec.T

dec_seq_len = decoder_input_ids.shape[1]
Q_dec_h = Q_dec.view(1, dec_seq_len, num_heads, d_kv).transpose(1, 2)
K_dec_h = K_dec.view(1, dec_seq_len, num_heads, d_kv).transpose(1, 2)
V_dec_h = V_dec.view(1, dec_seq_len, num_heads, d_kv).transpose(1, 2)

# Compute attention scores
dec_scores = torch.matmul(Q_dec_h, K_dec_h.transpose(-2, -1))

# Apply causal mask
causal_mask = torch.triu(torch.ones(dec_seq_len, dec_seq_len) * float("-inf"), diagonal=1)
dec_scores_masked = dec_scores + causal_mask.unsqueeze(0).unsqueeze(0)

dec_weights = F.softmax(dec_scores_masked, dim=-1)

print(f"\n  Causal attention weights (Head 0):")
dec_tokens = tokenizer.convert_ids_to_tokens(decoder_input_ids[0])
header = "           " + " ".join(f"{t[:6]:>7}" for t in dec_tokens)
print(header)
for i, row in enumerate(dec_weights[0, 0].detach().numpy()):
    row_str = f"  {dec_tokens[i][:8]:>8}  " + " ".join(f"{v:>7.3f}" for v in row)
    print(row_str)

print(f"\n  → Notice: upper triangle is 0 (future tokens are masked)")
print(f"    Position 0 can only see itself; position 4 can see all previous tokens")


---
## 10. Decoder Cross-Attention (Layer 0)

This is where encoder and decoder CONNECT. Cross-attention lets the decoder "look at" the encoder's output:

- **Query (Q)**: comes from the **decoder** ("what am I looking for?")
- **Key (K) & Value (V)**: come from the **encoder** ("what's available?")

```
Decoder hidden states → Q
Encoder output       → K, V

The decoder queries: "which encoder tokens are relevant for predicting the next German word?"
```


In [ ]:
print("=" * 70)
print("DECODER LAYER 0 — CROSS-ATTENTION (ENCODER-DECODER)")
print("=" * 70)

cross_attn = dec_layer0.layer[1].EncDecAttention

W_Q_cross = cross_attn.q.weight  # Query from decoder
W_K_cross = cross_attn.k.weight  # Key from encoder
W_V_cross = cross_attn.v.weight  # Value from encoder

print(f"\n  Cross-Attention weights:")
print(f"    W_Q (decoder → query):  {list(W_Q_cross.shape)}")
print(f"    W_K (encoder → key):    {list(W_K_cross.shape)}")
print(f"    W_V (encoder → value):  {list(W_V_cross.shape)}")

# Encoder output (from full encoder run)
encoder_output = encoder_outputs.last_hidden_state

with torch.no_grad():
    # Q comes from decoder
    cross_normed = dec_layer0.layer[1].layer_norm(decoder_embeddings)
    Q_cross = cross_normed @ W_Q_cross.T  # decoder query
    
    # K, V come from encoder
    K_cross = encoder_output @ W_K_cross.T  # encoder keys
    V_cross = encoder_output @ W_V_cross.T  # encoder values

print(f"\n  Q (from decoder): {list(Q_cross.shape)}  — {dec_seq_len} decoder positions querying")
print(f"  K (from encoder): {list(K_cross.shape)}  — {seq_len} encoder positions as keys")
print(f"  V (from encoder): {list(V_cross.shape)}  — {seq_len} encoder positions as values")

# Compute cross-attention scores
Q_cross_h = Q_cross.view(1, dec_seq_len, num_heads, d_kv).transpose(1, 2)
K_cross_h = K_cross.view(1, seq_len, num_heads, d_kv).transpose(1, 2)

cross_scores = torch.matmul(Q_cross_h, K_cross_h.transpose(-2, -1))
cross_weights = F.softmax(cross_scores, dim=-1)

print(f"\n  Cross-attention matrix shape: {list(cross_weights.shape)}")
print(f"    = (batch, heads, decoder_seq={dec_seq_len}, encoder_seq={seq_len})")
print(f"\n  Cross-attention weights (Head 0):")
print(f"  (rows=decoder tokens, cols=encoder tokens)")
enc_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
header = "           " + " ".join(f"{t[:6]:>7}" for t in enc_tokens)
print(header)
for i, row in enumerate(cross_weights[0, 0].detach().numpy()):
    row_str = f"  {dec_tokens[i][:8]:>8}  " + " ".join(f"{v:>7.3f}" for v in row)
    print(row_str)

print(f"\n  → This shows which English tokens each German token attends to!")
print(f"    No causal mask here — the decoder can see ALL encoder positions.")


---
## 11. Full Decoder: All 6 Layers

Each decoder layer has 3 sub-layers:
1. **Masked Self-Attention** (decoder attends to itself, causally)
2. **Cross-Attention** (decoder attends to encoder output)
3. **Feed-Forward Network** (per-position transformation)

Let's trace the full decoder and watch the tensor evolve.


In [ ]:
print("=" * 70)
print("FULL DECODER — TENSOR EVOLUTION ACROSS ALL 6 LAYERS")
print("=" * 70)

with torch.no_grad():
    decoder_outputs = model.decoder(
        input_ids=decoder_input_ids,
        encoder_hidden_states=encoder_output,
        output_hidden_states=True,
        output_attentions=True,
    )

print(f"\n{'Layer':<10} {'Shape':<25} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("─" * 75)

for i, hidden in enumerate(decoder_outputs.hidden_states):
    h = hidden[0]
    label = "Embed" if i == 0 else f"Layer {i}"
    print(f"  {label:<8} {str(list(hidden.shape)):<25} {h.mean():>10.4f} {h.std():>10.4f} {h.min():>10.4f} {h.max():>10.4f}")

print(f"\n  Decoder cross-attention attentions available: {len(decoder_outputs.cross_attentions)} layers")


In [ ]:
# Show cross-attention evolution: which encoder tokens matter most?
print("=" * 70)
print("CROSS-ATTENTION EVOLUTION — WHAT THE DECODER LOOKS AT")
print("=" * 70)

enc_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
dec_tokens = tokenizer.convert_ids_to_tokens(decoder_input_ids[0])

for layer_idx in [0, 2, 5]:
    cross_attn_weights = decoder_outputs.cross_attentions[layer_idx][0]  # (heads, dec_seq, enc_seq)
    # Average across heads
    avg_cross = cross_attn_weights.mean(dim=0)  # (dec_seq, enc_seq)
    
    print(f"\nLayer {layer_idx} — Each decoder token's top encoder attention (avg across heads):")
    for t_idx, tok in enumerate(dec_tokens):
        top_idx = avg_cross[t_idx].argmax().item()
        top_val = avg_cross[t_idx].max().item()
        print(f"  {tok:<12} → attends most to \"{enc_tokens[top_idx]}\" ({top_val:.3f})")


---
## 12. LM Head: Hidden States → Vocabulary Logits → Predicted Tokens

The final step: project the decoder's output from `d_model=512` dimensions to `vocab_size=32128` dimensions (one score per possible next token), then pick the highest-scoring token.

```
Decoder output (512-dim) → Linear(512, 32128) → logits → softmax → probabilities → argmax → token
```


In [ ]:
print("=" * 70)
print("LM HEAD — FROM HIDDEN STATES TO PREDICTED TOKENS")
print("=" * 70)

decoder_hidden = decoder_outputs.last_hidden_state
print(f"\nDecoder final hidden state: {list(decoder_hidden.shape)}")

# LM Head projection
lm_head = model.lm_head
print(f"LM Head weight: {list(lm_head.weight.shape)}  (vocab_size × d_model)")

with torch.no_grad():
    # T5 scales the output before the LM head
    # (In T5, the model scales by d_model^-0.5)
    logits = model.lm_head(decoder_hidden * (model.config.d_model ** -0.5))

print(f"Logits shape: {list(logits.shape)}  (batch, seq_len, vocab_size)")
print(f"  = one score for each of {logits.shape[-1]:,} possible tokens, at each position")

# Show predictions at each position
print(f"\n{'─' * 70}")
print("PREDICTIONS AT EACH POSITION")
print(f"{'─' * 70}")

probs = F.softmax(logits, dim=-1)

for pos in range(logits.shape[1]):
    top5_probs, top5_ids = probs[0, pos].topk(5)
    top5_tokens = [tokenizer.decode(tid.item()) for tid in top5_ids]
    
    actual = tokenizer.decode(target_ids[0, pos].item()) if pos < target_ids.shape[1] else "—"
    
    top_str = ", ".join(f"\"{t}\"({p:.1%})" for t, p in zip(top5_tokens, top5_probs))
    print(f"\n  Position {pos} (input: \"{dec_tokens[pos]}\")")
    print(f"    Target:  \"{actual}\"")
    print(f"    Top 5:   {top_str}")
    pred = tokenizer.decode(logits[0, pos].argmax().item())
    match = "✓" if pred.strip() == actual.strip() else "✗"
    print(f"    Predicted: \"{pred}\" {match}")


---
## 13. Full Autoregressive Generation

Now let's see the model **actually generate** a translation, token by token. At each step:
1. Run the encoder (once, for the input)
2. Feed the decoder what it's generated so far
3. Get logits for the next token
4. Pick the most likely token (greedy decoding)
5. Append it and repeat


In [ ]:
print("=" * 70)
print("AUTOREGRESSIVE GENERATION — TOKEN BY TOKEN")
print("=" * 70)

input_text = "translate English to German: The house is wonderful."
input_ids_gen = tokenizer(input_text, return_tensors="pt").input_ids

print(f"\nInput: \"{input_text}\"")
print(f"\nGenerating translation token by token...\n")

with torch.no_grad():
    # Step 1: Encode (done once)
    enc_out = model.encoder(input_ids=input_ids_gen)
    encoder_hidden = enc_out.last_hidden_state
    print(f"  Encoder output: {list(encoder_hidden.shape)}")
    
    # Start with just the pad token
    generated_ids = torch.tensor([[model.config.decoder_start_token_id]])
    
    print(f"\n  {'Step':<6} {'Input to decoder':<35} {'Logits shape':<25} {'Predicted':>15} {'Prob':>8}")
    print("  " + "─" * 95)
    
    for step in range(20):  # max 20 tokens
        # Step 2: Decode
        dec_out = model.decoder(
            input_ids=generated_ids,
            encoder_hidden_states=encoder_hidden,
        )
        
        # Step 3: Get logits for the LAST position only
        last_hidden = dec_out.last_hidden_state[:, -1, :]  # (1, 512)
        logits_step = model.lm_head(last_hidden * (model.config.d_model ** -0.5))  # (1, 32128)
        
        # Step 4: Pick the best token
        probs_step = F.softmax(logits_step, dim=-1)
        next_token_id = logits_step.argmax(dim=-1, keepdim=True)  # (1, 1)
        next_token_prob = probs_step[0, next_token_id[0, 0]].item()
        next_token = tokenizer.decode(next_token_id[0, 0].item())
        
        dec_input_str = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        print(f"  {step:<6} {dec_input_str:<35} {str(list(logits_step.shape)):<25} {next_token:>15} {next_token_prob:>7.1%}")
        
        # Step 5: Append and continue
        generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
        
        # Stop at EOS
        if next_token_id[0, 0].item() == tokenizer.eos_token_id:
            break

    final_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    print(f"\n  {'=' * 50}")
    print(f"  Input:       \"{input_text}\"")
    print(f"  Generated:   \"{final_text}\"")
    print(f"  Total steps: {step + 1}")
    print(f"  Final shape: {list(generated_ids.shape)}")


---
## 14. Verify: Official `model.generate()` Output

Let's verify our manual generation matches what the model officially produces.


In [ ]:
print("=" * 70)
print("VERIFICATION: model.generate() vs our manual generation")
print("=" * 70)

with torch.no_grad():
    official_output = model.generate(input_ids_gen, max_length=50)
    official_text = tokenizer.decode(official_output[0], skip_special_tokens=True)

print(f"\n  Our manual generation: \"{final_text}\"")
print(f"  Official generate():   \"{official_text}\"")
print(f"\n  Match: {'✓ Yes!' if final_text == official_text else '✗ Different (possibly due to generation config)'}")


---
## 15. Summary: All Weight Matrices at a Glance

Every trainable weight in the model — organized by where in the architecture they live.


In [ ]:
print("=" * 70)
print("ALL WEIGHT MATRICES IN T5-SMALL")
print("=" * 70)

categories = {
    "SHARED EMBEDDING": [(n, p) for n, p in model.named_parameters() if "shared" in n],
    "ENCODER SELF-ATTENTION": [(n, p) for n, p in model.named_parameters() 
                               if "encoder" in n and "SelfAttention" in n],
    "ENCODER FFN": [(n, p) for n, p in model.named_parameters() 
                    if "encoder" in n and "DenseRelu" in n],
    "ENCODER NORMS": [(n, p) for n, p in model.named_parameters() 
                      if "encoder" in n and "layer_norm" in n or "encoder.final_layer_norm" in n],
    "DECODER SELF-ATTENTION": [(n, p) for n, p in model.named_parameters() 
                               if "decoder" in n and "SelfAttention" in n],
    "DECODER CROSS-ATTENTION": [(n, p) for n, p in model.named_parameters() 
                                if "decoder" in n and "EncDecAttention" in n],
    "DECODER FFN": [(n, p) for n, p in model.named_parameters() 
                    if "decoder" in n and "DenseRelu" in n],
    "LM HEAD": [(n, p) for n, p in model.named_parameters() if "lm_head" in n],
}

for cat_name, params in categories.items():
    if not params:
        continue
    total = sum(p.numel() for _, p in params)
    print(f"\n  {cat_name} ({total:,} params)")
    print(f"  {'─' * 60}")
    for name, p in params[:6]:  # show first 6
        short_name = name.replace("model.", "")
        print(f"    {short_name:<50} {str(list(p.shape)):>15}")
    if len(params) > 6:
        print(f"    ... and {len(params) - 6} more")

total_all = sum(p.numel() for p in model.parameters())
print(f"\n  {'=' * 60}")
print(f"  TOTAL: {total_all:,} parameters")


---
## 🎯 Summary: The Complete Token Journey

```
INPUT TEXT: "translate English to German: The house is wonderful."
                                    │
                            ┌───────▼────────┐
                            │  Tokenizer     │  "translate" → 13959
                            │  (SentencePiece)│  "English"  → 1566
                            └───────┬────────┘  ...
                                    │
                            ┌───────▼────────┐
                            │  Embedding     │  13959 → [0.02, -0.1, ...] (512-dim)
                            │  Lookup Table  │
                            └───────┬────────┘
                                    │
                    ┌───────────────▼───────────────────┐
                    │         ENCODER (×6 layers)        │
                    │  ┌──────────────────────────────┐  │
                    │  │ LayerNorm → Self-Attention    │  │
                    │  │   Q, K, V from same input     │  │
                    │  │   8 heads × 64 dims           │  │
                    │  │   + relative position bias    │  │
                    │  │   + residual connection       │  │
                    │  ├──────────────────────────────┤  │
                    │  │ LayerNorm → FFN               │  │
                    │  │   512 → 2048 → ReLU → 512    │  │
                    │  │   + residual connection       │  │
                    │  └──────────────────────────────┘  │
                    └───────────────┬───────────────────┘
                                    │ encoder_output (seq_len, 512)
                                    │
                    ┌───────────────▼───────────────────┐
                    │         DECODER (×6 layers)        │
                    │  ┌──────────────────────────────┐  │
                    │  │ Masked Self-Attention         │  │
                    │  │   Q, K, V from decoder input  │  │
                    │  │   (causal mask: no future)    │  │
                    │  ├──────────────────────────────┤  │
                    │  │ Cross-Attention               │  │
                    │  │   Q from decoder              │  │
                    │  │   K, V from ENCODER output    │  │
                    │  ├──────────────────────────────┤  │
                    │  │ FFN                           │  │
                    │  │   512 → 2048 → ReLU → 512    │  │
                    │  └──────────────────────────────┘  │
                    └───────────────┬───────────────────┘
                                    │ decoder_output (1, 512)
                            ┌───────▼────────┐
                            │  LM Head       │  512 → 32128 logits
                            │  + softmax     │  → probabilities
                            │  + argmax      │  → token ID
                            └───────┬────────┘
                                    │
                            ┌───────▼────────┐
                            │  Detokenize    │  token ID → "Das"
                            └────────────────┘

OUTPUT: "Das Haus ist wunderbar."
```

### Key Takeaways:
1. **Embeddings** convert discrete tokens to continuous vectors
2. **Encoder self-attention** lets every input token see every other input token
3. **Decoder masked self-attention** only lets tokens see PAST tokens (causal)
4. **Cross-attention** connects decoder to encoder (Q from decoder, K/V from encoder)
5. **FFN** processes each position independently (no inter-token interaction)
6. **Residual connections** at every sub-layer keep gradients flowing
7. **Layer norms** stabilize training
8. **LM Head** projects back to vocabulary size for token prediction
